## Creamos SparkContext y SarkSession

In [1]:
import os
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pathlib import Path

import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

conf = (SparkConf()
        .setMaster("local[*]")        # Corre Spark en tu máquina local, usando todos los núcleos disponibles del CPU ("local" a secas, corre Spark en modo local con un solo hilo (1 CPU))
        .setAppName("5.1 Spark y MapReduce")
        .set("spark.driver.bindAddress", "127.0.0.1")
        .set("spark.driver.host", "127.0.0.1"))
sc = SparkContext(conf = conf)



spark = SparkSession(sc)

sc, spark

(<SparkContext master=local[*] appName=5.1 Spark y MapReduce>,
 <pyspark.sql.session.SparkSession at 0x25c99aeadb0>)

## CONSULTA 1
### Buscamos los 5 clientes que mas gastaron
#### Leemos el csv

In [7]:

BASE_DIR = Path.cwd()

ruta_csv = BASE_DIR.parent / "csv" / "pagos_clientes.csv"
df_payment = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

df_payment.show(5)

+-----------+------+
|customer_id|amount|
+-----------+------+
|       3330| 78.62|
|       1503| 31.08|
|        150| 28.75|
|       3207| 58.44|
|       3546| 26.35|
+-----------+------+
only showing top 5 rows


#### Aplicamos MAP SHUFFLE REDUCE

In [8]:
rdd = df_payment.rdd

resultado = (
    rdd
    .map(lambda row: (row.customer_id, row.amount))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1] , ascending=False)
)

resultado.take(5)

[(3816, 854.8299999999999),
 (4288, 843.48),
 (440, 841.3199999999999),
 (2402, 836.78),
 (2109, 827.0300000000001)]

## CONSULTA 2
### Buscamos las peliculas que mas se rentan juntas (de a pares)
#### Leemos el csv

In [5]:
BASE_DIR = Path.cwd()

ruta_csv = BASE_DIR.parent / "csv" / "rental_inventory.csv"
df_ri = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

df_ri.show(5)

+---------+------------+
|rental_id|inventory_id|
+---------+------------+
|        1|        1611|
|        1|        5613|
|        1|        3250|
|        1|        1188|
|        1|        3902|
+---------+------------+
only showing top 5 rows


#### Aplicamos MAP SHUFFLE REDUCE

In [10]:
rdd = df_ri.rdd

def combinar(iterable):
    lista =list(iterable)
    combinaciones = []
    for i in range(len(lista)-1):
        for j in range(i+1,len(lista)):
            combinaciones.append(((lista[i],lista[j]),1))
    
    return combinaciones

#agrupamos las peliculas por rentas
rentas = rdd.map(lambda row: (row.rental_id,row.inventory_id)).groupByKey()
# obtenemos k: renta_id v:inventory_id[]
print("obtenemos k: renta_id v:inventory_id[]")
print(rentas.take(5))

combinaciones = rentas.flatMap(lambda x: combinar(x[1]))
print("obtenemos k: (inventory_id,inventory_id) v:1")
print(combinaciones.take(5))

resultado_final = combinaciones.reduceByKey(lambda x , y: x+y).sortBy(lambda x: x[1], ascending=False)

print("resultado_final :", resultado_final.take(10))


obtenemos k: renta_id v:inventory_id[]
[(1, <pyspark.resultiterable.ResultIterable object at 0x0000025C9B26F650>), (2, <pyspark.resultiterable.ResultIterable object at 0x0000025C9B3004A0>), (3, <pyspark.resultiterable.ResultIterable object at 0x0000025C9B301040>), (4, <pyspark.resultiterable.ResultIterable object at 0x0000025C9B300F50>), (5, <pyspark.resultiterable.ResultIterable object at 0x0000025C9B300B90>)]
obtenemos k: (inventory_id,inventory_id) v:1
[((1611, 5613), 1), ((1611, 3250), 1), ((1611, 1188), 1), ((1611, 3902), 1), ((5613, 3250), 1)]
resultado_final : [((5071, 939), 3), ((1147, 1368), 3), ((5860, 2963), 3), ((1504, 238), 2), ((3102, 5006), 2), ((5736, 1063), 2), ((4972, 1653), 2), ((2318, 4984), 2), ((111, 2849), 2), ((756, 479), 2)]


## Consulta 3
### ¿Cual es el ingreso promedio por metodo de pago y cuantas transacciones tuvo cada uno?
#### Leer csv

In [6]:
BASE_DIR = Path.cwd()
ruta_csv = BASE_DIR.parent / "csv" / "pay_method_amount.csv"

df_pm_a = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

df_pm_a.show(5)

+-------------+------+
|pay_method_id|amount|
+-------------+------+
|            3| 40.68|
|            4| 88.87|
|            3| 25.63|
|            1| 10.48|
|            4|  35.2|
+-------------+------+
only showing top 5 rows


In [14]:
rdd = df_pm_a.rdd

mapeo = rdd.map(lambda row: (row.pay_method_id,(row.amount,1)))

print("obtenemos k: pay_method_id , v: (amount,1)")
print("mapeo: ", mapeo.take(5))
print()
pre_resultado = mapeo.reduceByKey(lambda x, y : (x[0]+y[0],x[1]+y[1]))

print("obtenemos k: pay_method_id , v: (sum(amount),n)")
print("Pre resultado:", pre_resultado.take(5))
print()

resultado = pre_resultado.map(lambda tupla: (tupla[0], tupla[1][0]/tupla[1][1]))
print("obtenemos tupla(metodo_de_pago_id,importe_promedio)")
print("resultado: ", resultado.take(5))


obtenemos k: pay_method_id , v: (amount,1)
mapeo:  [(3, (40.68, 1)), (4, (88.87, 1)), (3, (25.63, 1)), (1, (10.48, 1)), (4, (35.2, 1))]

obtenemos k: pay_method_id , v: (sum(amount),n)
Pre resultado: [(3, (192027.31999999975, 3836)), (4, (182932.45000000016, 3727)), (1, (183635.98000000024, 3679)), (2, (186650.36000000022, 3758))]

obtenemos tupla(metodo_de_pago_id,importe_promedio)
resultado:  [(3, 50.05925964546396), (4, 49.083029246042436), (1, 49.914645284044646), (2, 49.66747205960623)]
